# Assign MolID to the entities found in the CIF files (1) 

## What is the goal of this notebook?

We will run `CIFcurator_MolID.py` to re-assign the MolID to the entities found in our new ensemble of CIF files. 
The script goes over all the CIF files and collects all entities. The user can then decide what MolID to assign them. 

There are also some other benefits from running this script: 

- You can assign the same MolID to different entities. In that case these entities will be concatenated. User needs to accept each concatenation manually. 
- Inspecting the list of entities will allow users to identify structures that needs to be removed from the ensemble.
- Make sure that the MolIDs of the structures in the ensemble are consistent (the same chain is named always the same, even in different structures).

This notebook will go over the cases described above. 

>**NOTE:** For this tutorial, we will not use the whole ensemble we downloaded. We will use a subsample of only 7 structures. The next cells will create the new directory. Notice that we are choosing these 7 sctructures from the ones we downloaded. We chose these ones to highlight some possible issues you may run into when running this script.

In [1]:
from CIFcurator import cifcurator_io

In [2]:
PROJDIR="./TIM/"

In [3]:
cifcurator_io.check_project(projdir=PROJDIR, action='create', level='clean_bank_sub')

> Let's copy some structures from our clean_bank into the newly created 'clean_bank_sub' directory

In [4]:
!cp $PROJDIR/clean_bank/pdb_00001klg+01.cif $PROJDIR/clean_bank_sub/
!cp $PROJDIR/clean_bank/pdb_00002y62+01.cif $PROJDIR/clean_bank_sub/
!cp $PROJDIR/clean_bank/pdb_00001ag1+01.cif $PROJDIR/clean_bank/pdb_00001aw1+01.cif $PROJDIR/clean_bank_sub/
!cp $PROJDIR/clean_bank/pdb_00001aw1+02.cif $PROJDIR/clean_bank/pdb_00001aw1+03.cif $PROJDIR/clean_bank_sub/
!cp $PROJDIR/clean_bank/pdb_00001aw1+04.cif $PROJDIR/clean_bank_sub/
!cp $PROJDIR/clean_bank/AF-B5YL53-F1+01.cif $PROJDIR/clean_bank_sub/
!cp $PROJDIR/clean_bank/AF-B8E2R2-F1+01.cif $PROJDIR/clean_bank_sub/

In [5]:
cifcurator_io.check_project(projdir=PROJDIR, action='create', level='standard_MolID_bank')

## Running CIFcurator_MolID.py 

Notice that the way to run this script in the terminal is as following:

> CIFcurator_MolID.py `{Input Directory}` `{Output Directory}`

The input directory contains the structures that we generated in Step 1. The output directory is where the new structures will be stored. 

Running this script will print a menu to screen. In the next cell we run the script and give 2 as input, so that we can select option `2) Show only unassigned conversions`. Then we `QUIT` the program. 

**Note:** We recommend running the script directly on the terminal. We are running it from the notebook just for demonstration purpose.

In [6]:
! echo '2\n4\nQUIT' | CIFcurator_MolID.py $PROJDIR/clean_bank_sub $PROJDIR/standard_MolID_bank $PROJDIR $PROJDIR/seqres.txt

Directory './TIM//clustering_storage/' created successfully.


Processing PROTEIN: 15 sequences
Cluster Stage: PROTEIN
cwd: /Users/fpardo/tmp/Notebooks
entity_seqres: /Users/fpardo/tmp/Notebooks/TIM/clustering_storage/PROTEIN_clustering_storage/PROTEIN_seqres.txt
absolute: /Users/fpardo/tmp/Notebooks/TIM/clustering_storage/PROTEIN_clustering_storage/PROTEIN_seqres.txt
exists: True
Creating MMseqs2 database...
createdb /Users/fpardo/tmp/Notebooks/TIM/clustering_storage/PROTEIN_clustering_storage/PROTEIN_seqres.txt pdbDB 

MMseqs Version:       	15.6f452
Database type         	0
Shuffle input database	true
Createdb mode         	0
Write lookup file     	1
Offset of numeric ids 	0
Compressed            	0
Verbosity             	3

Converting sequences
[
Time for merging to pdbDB_h: 0h 0m 0s 2ms
Time for merging to pdbDB: 0h 0m 0s 2ms
Database type: Aminoacid
Time for processing: 0h 0m 0s 12ms
Clustering sequences...
Create directory tmp
cluster pdbDB clusterDB tmp --min-seq-id 0.30 --clus

## What does the output printed to screen mean?
```
2:TRIOSEPHOSPHATE ISOMERASE:
1:HLA CLASS II HISTOCOMPATIBILITY ANTIGEN, DR ALPHA CHAIN:
1:HLA CLASS II HISTOCOMPATIBILITY ANTIGEN, DR-1 BETA CHAIN:
1:TRIOSEPHOSPHATE ISOMERASE PEPTIDE:
1:ENTEROTOXIN TYPE C-3:
4:WATER:
2:TRIOSEPHOSPHATE ISOMERASE SYNONYM TRIOSE-PHOSPHATE ISOMERASE, TIM:
2:SN-GLYCEROL-3-PHOSPHATE:
2:SN-GLYCEROL-1-PHOSPHATE:
4:GLYCEROL:
1:PHOSPHATE ION:
2:2-PHOSPHOGLYCOLIC ACID:
You need to accept 12 entity conversions
You need to accept 23 total chain conversions
```

The output printed to the screen, and reproduced right above in this cell, tells us how many chains are part of each entity. For example, the first line tells us that in one of the files, there is one entity `TRIOSEPHOSPHATE ISOMERASE` that contains two chains. We also see that, for `WATER`, there are 4 chain names to assign. 

The last two lines tell us how many entities were found and the total number of chains.

## Generate input file

Selecting option 4 automatically creates the file input_file.txt. This files contains automatically generated chain assignments. These assignments are selected based on the results of the sequence clustering of all the chains in the dataset. Below, we inspect the input_file.txt generated in the previous cell.

In [7]:
! cat $PROJDIR/input_file.txt

2-PHOSPHOGLYCOLIC ACID:K,K
ENTEROTOXIN TYPE C-3:E
GLYCEROL:I,I,I,I
HLA CLASS II HISTOCOMPATIBILITY ANTIGEN, DR ALPHA CHAIN:B
HLA CLASS II HISTOCOMPATIBILITY ANTIGEN, DR-1 BETA CHAIN:C
PHOSPHATE ION:J
SN-GLYCEROL-1-PHOSPHATE:H,H
SN-GLYCEROL-3-PHOSPHATE:G,G
TRIOSEPHOSPHATE ISOMERASE:A,Ab
TRIOSEPHOSPHATE ISOMERASE PEPTIDE:D
TRIOSEPHOSPHATE ISOMERASE SYNONYM TRIOSE-PHOSPHATE ISOMERASE, TIM:A,Ab
WATER:F,F,F,F


## Inspect the entities in your ensemble. A way to detect outliers:

Another advantage of reading this list, is that we can take a look at all the entities that are present in our ensemble. In our tutorial example, we used the keyword 'triosephosphate isomerase'. If you read this list, you may find some suspicious entitities, such as `HLA CLASS II HISTOCOMPATIBILITY ANTIGEN, DR ALPHA CHAIN`. A closer inspection to the list, we can see also `TRIOSEPHOSPHATE ISOMERASE PEPTIDE`, which suggests that it only contains a fragment of the protein. Furthermore, `TRIOSEPHOSPHATE ISOMERASE PEPTIDE` is assigned a different chain name compared to both `TRIOSEPHOSPHATE ISOMERASE` and `TRIOSEPHOSPHATE ISOMERASE SYNONYM TRIOSE-PHOSPHATE ISOMERASE`.

Since these are suspicious entries, we can further inspect the CIF files that contain these entities. First, we need to figure out which are the CIF files. The next cell shows a way to do it:

In [8]:
! grep "HLA CLASS II HISTOCOMPATIBILITY ANTIGEN" $PROJDIR/clean_bank_sub/*cif 
! grep "TRIOSEPHOSPHATE ISOMERASE PEPTIDE" $PROJDIR/clean_bank_sub/*cif 

./TIM//clean_bank_sub/pdb_00001klg+01.cif:1 polymer man 'HLA CLASS II HISTOCOMPATIBILITY ANTIGEN, DR ALPHA CHAIN'
./TIM//clean_bank_sub/pdb_00001klg+01.cif:2 polymer man 'HLA CLASS II HISTOCOMPATIBILITY ANTIGEN, DR-1 BETA CHAIN'
./TIM//clean_bank_sub/pdb_00001klg+01.cif:3 polymer syn 'TRIOSEPHOSPHATE ISOMERASE PEPTIDE'


These entities come from one single CIF file: pdb_00001klg+00.cif 

By reading the CIF file (run the cell below, removing the '#') , or using a molecular visualization tool, the user can see that this is an outlier. It was selected because there is a small fragment of the triosephosphate isomerase, but the main structure is of the HLA Class II Histocompatibility antigen. It is best to remove these structures from our ensemble. 

In [9]:
# ! cat $PROJDIR/clean_bank_sub/pdb_00001klg+00.cif

In [10]:
# Remove problematic CIF file

! rm $PROJDIR/clean_bank_sub/pdb_00001klg+01.cif 


## Running CIFcurator_MolID.py with the new subset

Let's rerun `CIFcurator_MolID.py` with our subsampled ensemble, now with only 6 structures. 

In [11]:
!printf '4\n3\n%s\n7\n5\n6\n' "$PROJDIR/input_file.txt" | CIFcurator_MolID.py $PROJDIR/clean_bank_sub $PROJDIR/standard_MolID_bank $PROJDIR $PROJDIR/seqres.txt

Directory './TIM//clustering_storage/' already exists.

Processing PROTEIN: 11 sequences
Cluster Stage: PROTEIN
cwd: /Users/fpardo/tmp/Notebooks
entity_seqres: /Users/fpardo/tmp/Notebooks/TIM/clustering_storage/PROTEIN_clustering_storage/PROTEIN_seqres.txt
absolute: /Users/fpardo/tmp/Notebooks/TIM/clustering_storage/PROTEIN_clustering_storage/PROTEIN_seqres.txt
exists: True
Creating MMseqs2 database...
createdb /Users/fpardo/tmp/Notebooks/TIM/clustering_storage/PROTEIN_clustering_storage/PROTEIN_seqres.txt pdbDB 

MMseqs Version:       	15.6f452
Database type         	0
Shuffle input database	true
Createdb mode         	0
Write lookup file     	1
Offset of numeric ids 	0
Compressed            	0
Verbosity             	3

Converting sequences
[
Time for merging to pdbDB_h: 0h 0m 0s 2ms
Time for merging to pdbDB: 0h 0m 0s 2ms
Database type: Aminoacid
Time for processing: 0h 0m 0s 11ms
Clustering sequences...
Create directory tmp
cluster pdbDB clusterDB tmp --min-seq-id 0.30 --cluster-mod

## Understanding the output

When we first run CIFcurator_MolID.py, we automatically cluster all the chains of the cif structures inside the input folder. The clustering process is printed to screen, as well as the progress loading the structure files. 

Now, let's inspect the automatically generated `input_file.txt`, created after removing the outlier structure.

In [12]:
! cat $PROJDIR/input_file.txt

2-PHOSPHOGLYCOLIC ACID:G,G
GLYCEROL:D,D,D,D
PHOSPHATE ION:F
SN-GLYCEROL-1-PHOSPHATE:C,C
SN-GLYCEROL-3-PHOSPHATE:B,B
TRIOSEPHOSPHATE ISOMERASE:A,Ab
TRIOSEPHOSPHATE ISOMERASE SYNONYM TRIOSE-PHOSPHATE ISOMERASE, TIM:A,Ab
WATER:E,E


We were able to identify that both `TRIOSEPHOSPHATE ISOMERASE` and `TRIOSEPHOSPHATE ISOMERASE SYNONYM TRIOSE-PHOSPHATE ISOMERASE, TIM` refer to the same type of entity, even if their names are slightly different. In both cases, we have at most 2 chains in a single file; thus, we assigned the base name A to the first chain and added the suffix b to the second chain. 

We do not automatically move on to the next step (changing chain names), as we want to allow users to inspect whether they are satisfied with the automatic name assignments. If they are, they can proceed by inputting the input_file.txt.

In the case of ligands and water, we propose merging each type of entity into a single chain. For example, we assigned chain E to `WATER` and chain D to `GLYCEROL`. As a result, a "merging menu" appears after you input input_file.txt.

After accepting the merge, all the chains that are merged are printed to the screen. We suggest that users inspect this output to prevent any unwanted chains or entities from being merged. 

Choosing option 6 finalizes this step. 

We also automatically save a file to the output directory that tracks changes to chain names:  OldChainID_NewChainID_numbered_MolID_Files_Map.csv

In [14]:
! cat $PROJDIR/standard_MolID_bank/OldChainID_NewChainID_numbered_MolID_Files_Map.csv

OldChainID(label_asym_id):NewChainID:Entity:NumberOfFiles:Files
A:A:TRIOSEPHOSPHATE ISOMERASE:7:AF-B5YL53-F1+01.cif
A,B:A,Ab:TRIOSEPHOSPHATE ISOMERASE:7:pdb_00001ag1+01.cif
G,H:A,Ab:TRIOSEPHOSPHATE ISOMERASE:7:pdb_00001aw1+04.cif
A,B:A,Ab:TRIOSEPHOSPHATE ISOMERASE:7:pdb_00001aw1+01.cif
E,F:A,Ab:TRIOSEPHOSPHATE ISOMERASE:7:pdb_00001aw1+03.cif
C,D:A,Ab:TRIOSEPHOSPHATE ISOMERASE:7:pdb_00001aw1+02.cif
A:A:TRIOSEPHOSPHATE ISOMERASE:7:AF-B8E2R2-F1+01.cif
A,A-2:A,Ab:TRIOSEPHOSPHATE ISOMERASE SYNONYM TRIOSE-PHOSPHATE ISOMERASE, TIM:1:pdb_00002y62+01.cif
B,B-2:B,B:SN-GLYCEROL-3-PHOSPHATE:1:pdb_00002y62+01.cif
C,C-2:C,C:SN-GLYCEROL-1-PHOSPHATE:1:pdb_00002y62+01.cif
D,E,D-2,E-2:D,D,D,D:GLYCEROL:1:pdb_00002y62+01.cif
F,F-2:E,E:WATER:6:pdb_00002y62+01.cif
D,E:E,E:WATER:6:pdb_00001ag1+01.cif
W,X:E,E:WATER:6:pdb_00001aw1+04.cif
Q,R:E,E:WATER:6:pdb_00001aw1+01.cif
U,V:E,E:WATER:6:pdb_00001aw1+03.cif
S,T:E,E:WATER:6:pdb_00001aw1+02.cif
C:F:PHOSPHATE ION:1:pdb_00001ag1+01.cif
O,P:G,G:2-PHOSPHOGLYCOLIC A